In [4]:
import pandas as pd
from pathlib import Path
import requests
import tarfile
from wikimapper import WikiMapper
from datapackage import Package
import numpy as np

In [5]:
import os

wikidata_df = pd.read_csv("movie.metadata.tsv", sep='\t')
wikidata_df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'movie.metadata.tsv'

In [ ]:
import os
import zipfile
import glob
import shutil

# ------------------------------
# Step 1: Download the dataset
# ------------------------------
!kaggle datasets download -d bahramjannesarr/goodreads-book-datasets-10m --force

# ------------------------------
# Step 2: Unzip the dataset
# ------------------------------
with zipfile.ZipFile("goodreads-book-datasets-10m.zip", 'r') as zip_ref:
    zip_ref.extractall("data/goodreads_temp")

# ------------------------------
# Step 3: Remove the zip file
# ------------------------------
os.remove("goodreads-book-datasets-10m.zip")

# ------------------------------
# Step 4: Remove unwanted user_rating CSVs
# ------------------------------
for f in glob.glob("data/goodreads_temp/user_rating_*.csv"):
    os.remove(f)

# ------------------------------
# Step 5: Move book CSVs to final folder (skip if already exists)
# ------------------------------
os.makedirs("data/goodreads", exist_ok=True)
for f in glob.glob("data/goodreads_temp/book*.csv"):
    dest = os.path.join("data/goodreads", os.path.basename(f))
    if not os.path.exists(dest):
        shutil.move(f, "data/goodreads/")

# ------------------------------
# Step 6: Remove the temporary folder safely
# ------------------------------
shutil.rmtree("data/goodreads_temp")

print("Goodreads dataset prepared successfully!")



  0%|          | 0.00/460M [00:00<?, ?B/s]
 18%|█▊        | 83.0M/460M [00:00<00:00, 858MB/s]
 39%|███▉      | 180M/460M [00:00<00:00, 912MB/s] 
 58%|█████▊    | 267M/460M [00:00<00:00, 879MB/s]
 76%|███████▌  | 351M/460M [00:00<00:00, 766MB/s]
 93%|█████████▎| 426M/460M [00:00<00:00, 707MB/s]
100%|██████████| 460M/460M [00:00<00:00, 757MB/s]


Dataset URL: https://www.kaggle.com/datasets/bahramjannesarr/goodreads-book-datasets-10m
License(s): CC0-1.0

Goodreads dataset prepared successfully!


In [ ]:
def clean_title(title_series: pd.Series) -> pd.Series:
    return (title_series
            .str.split('(').str[0]
            .str.split(':').str[0]
            .str.lower()
            .str.replace('and', '&')
            .str.replace('.', '')
            .str.replace("'", '')
            .str.replace('-', ' ')
            .str.replace(r'\s+', ' ', regex=True)
            .str.strip()
    )

def clean_author(author_series: pd.Series) -> pd.Series:
    initial_letter = (author_series
                      .str.strip()
                      .str[0]
                      .str.lower())
    last_name = (author_series
                 .str.split(r"(\s|-|')", regex=True)
                 .str[-1]
                 .str.replace('.', '')
                 .str.replace("'", '')
                 .str.replace(r'\s+', ' ', regex=True)
                 .str.strip()
                 .str.lower()
                 )
    return initial_letter + " " + last_name

In [ ]:
print(len(wikidata_df.columns))
print(wikidata_df.columns.tolist())


11
['book_id', 'author_id', 'book_title', 'publication_date', 'goodreads_id', 'rating', 'language', 'country', 'genres', 'join_title', 'join_author']


In [ ]:
# Step 1: Rename columns
wikidata_df.columns = [
    'book_id', 'author_id', 'book_title', 'publication_date', 
    'goodreads_id', 'rating', 'language', 'country', 'genres', 'join_title', 'join_author'
]

# Step 2: Create cleaned title and author columns
# Note: If you don’t have author names, you might skip join_author or use author_id temporarily
wikidata_df = wikidata_df.assign(
    join_title = lambda x: clean_title(x['book_title']),
    join_author = lambda x: clean_author(x['author_id'])  # Replace with author name if available
)

# Check result
wikidata_df[['book_title', 'join_title', 'author_id', 'join_author']].head()


,book_title,join_title,author_id,join_author
0,Getting Away with Murder: The JonBenét Ramsey ...,getting away with murder,/m/08yl5d,/ /m/08yl5d
1,Brun bitter,brun bitter,/m/0crgdbh,/ /m/0crgdbh
2,White Of The Eye,white of the eye,/m/0285_cd,/ /m/0285_cd
3,A Woman in Flames,a woman in flames,/m/01mrr1,/ /m/01mrr1
4,The Gangsters,the gangsters,/m/03cfc81,/ /m/03cfc81


In [ ]:
with open("book1-100k.csv", "r", encoding="utf-8") as f:
    for _ in range(10):
        print(f.readline())


Id,Name,RatingDist1,pagesNumber,RatingDist4,RatingDistTotal,PublishMonth,PublishDay,Publisher,CountsOfReview,PublishYear,Language,Authors,Rating,RatingDist2,RatingDist5,ISBN,RatingDist3

1,"Harry Potter and the Half-Blood Prince (Harry Potter, #6)",1:9896,652,4:556485,total:2298124,16,9,Scholastic Inc.,28062,2006,eng,J.K. Rowling,4.57,2:25317,5:1546466,,3:159960

2,"Harry Potter and the Order of the Phoenix (Harry Potter, #5)",1:12455,870,4:604283,total:2358637,1,9,Scholastic Inc.,29770,2004,eng,J.K. Rowling,4.5,2:37005,5:1493113,0439358078,3:211781

3,"Harry Potter and the Sorcerer's Stone (Harry Potter, #1)",1:108202,309,4:1513191,total:6587388,1,11,Scholastic Inc,75911,2003,eng,J.K. Rowling,4.47,2:130310,5:4268227,,3:567458

4,"Harry Potter and the Chamber of Secrets (Harry Potter, #2)",1:11896,352,4:706082,total:2560657,1,11,Scholastic,244,2003,eng,J.K. Rowling,4.42,2:49353,5:1504505,0439554896,3:288821

5,"Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)",1:10128,435,4:

In [ ]:
book_df = pd.read_csv("book1-100k.csv", encoding="utf-8")
book_df.head()


,Id,Name,RatingDist1,pagesNumber,RatingDist4,RatingDistTotal,PublishMonth,PublishDay,Publisher,CountsOfReview,PublishYear,Language,Authors,Rating,RatingDist2,RatingDist5,ISBN,RatingDist3
0,1,Harry Potter and the Half-Blood Prince (Harry ...,1:9896,652,4:556485,total:2298124,16,9,Scholastic Inc.,28062,2006,eng,J.K. Rowling,4.57,2:25317,5:1546466,NaN,3:159960
1,2,Harry Potter and the Order of the Phoenix (Har...,1:12455,870,4:604283,total:2358637,1,9,Scholastic Inc.,29770,2004,eng,J.K. Rowling,4.50,2:37005,5:1493113,0439358078,3:211781
2,3,Harry Potter and the Sorcerer's Stone (Harry P...,1:108202,309,4:1513191,total:6587388,1,11,Scholastic Inc,75911,2003,eng,J.K. Rowling,4.47,2:130310,5:4268227,NaN,3:567458
3,4,Harry Potter and the Chamber of Secrets (Harry...,1:11896,352,4:706082,total:2560657,1,11,Scholastic,244,2003,eng,J.K. Rowling,4.42,2:49353,5:1504505,0439554896,3:288821
4,5,Harry Potter and the Prisoner of Azkaban (Harr...,1:10128,435,4:630534,total:2610317,1,5,Scholastic Inc.,37093,2004,eng,J.K. Rowling,4.57,2:24849,5:1749958,043965548X,3:194848


In [ ]:
book_df = book_df.assign(
    join_title = lambda x: clean_title(x['Name']),
    join_author = lambda x: clean_author(x['Authors'])
)

book_df[['Name', 'join_title', 'Authors', 'join_author']].head()

,Name,join_title,Authors,join_author
0,Harry Potter and the Half-Blood Prince (Harry ...,harry potter & the half blood prince,J.K. Rowling,j rowling
1,Harry Potter and the Order of the Phoenix (Har...,harry potter & the order of the phoenix,J.K. Rowling,j rowling
2,Harry Potter and the Sorcerer's Stone (Harry P...,harry potter & the sorcerers stone,J.K. Rowling,j rowling
3,Harry Potter and the Chamber of Secrets (Harry...,harry potter & the chamber of secrets,J.K. Rowling,j rowling
4,Harry Potter and the Prisoner of Azkaban (Harr...,harry potter & the prisoner of azkaban,J.K. Rowling,j rowling


In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("data")  # adjust if your folder is different
df_list = []

for file in data_path.glob('goodreads/book*.csv'):
    try:
        df = pd.read_csv(file)
        # Normalize column names
        df = df.rename(columns=lambda x: x.strip())  # remove leading/trailing spaces
        df = df.rename(columns={
            'pagesNumber': 'book_pages',
            'PagesNumber': 'book_pages',
            'RatingDistTotal': 'book_ratings_count',
            'Rating': 'book_rating',
            'Publisher': 'book_publisher',
            'Authors': 'book_author',
            'Name': 'book_title'
        })
        # Keep only the columns we care about
        cols_to_keep = ['book_title', 'book_author', 'book_publisher', 'book_pages', 'book_rating', 'book_ratings_count']
        df = df[[c for c in cols_to_keep if c in df.columns]]
        df_list.append(df)
    except Exception as e:
        print(f"Failed to process {file}: {e}")

# Combine all files into a single DataFrame
goodreads_df = pd.concat(df_list, ignore_index=True)
goodreads_df.head()


,book_title,book_author,book_publisher,book_pages,book_rating,book_ratings_count
0,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling,Scholastic Inc.,652,4.57,total:2298124
1,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling,Scholastic Inc.,870,4.50,total:2358637
2,Harry Potter and the Sorcerer's Stone (Harry P...,J.K. Rowling,Scholastic Inc,309,4.47,total:6587388
3,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,Scholastic,352,4.42,total:2560657
4,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling,Scholastic Inc.,435,4.57,total:2610317


In [ ]:
goodreads_df = (pd.concat(df_list, ignore_index=True)
                .assign(
                    book_rating = lambda x: x.book_rating.replace(0, np.nan),
                    book_pages = lambda x: x.book_pages.replace(0, np.nan).astype('Int64'),
                    book_ratings_count = lambda x: x.book_ratings_count.str.split(':').str[-1].astype('Int64'),
                    join_title = lambda x: clean_title(x.book_title),
                    join_author = lambda x: clean_author(x.book_author), 
                )
                .merge(
                    wikidata_df.loc[:, ['join_title', 'join_author']].drop_duplicates(), 
                    on=['join_title', 'join_author'], 
                    how='inner'
                    )
                .drop_duplicates(subset=['join_title', 'join_author'])
                .drop(['book_title', 'book_author'], axis=1)
            )

In [ ]:
goodreads_book_df = (pd.concat(df_list, ignore_index=True)
                .assign(
                    book_rating = lambda x: x.book_rating.replace(0, np.nan),
                    book_pages = lambda x: x.book_pages.replace(0, np.nan).astype('Int64'),
                    book_ratings_count = lambda x: x.book_ratings_count.str.split(':').str[-1].astype('Int64'),
                    join_title = lambda x: clean_title(x.book_title),
                    join_author = lambda x: clean_author(x.book_author), 
                )
                .merge(
                    book_df.loc[:, ['join_title', 'join_author']].drop_duplicates(), 
                    on=['join_title', 'join_author'], 
                    how='inner'
                    )
                .drop_duplicates(subset=['join_title', 'join_author'])
                .drop(['book_title', 'book_author'], axis=1)
            )

In [ ]:
import os
import zipfile
import shutil

# Step 1: Download the dataset (requires Kaggle API installed and configured)
!kaggle datasets download -d rounakbanik/the-movies-dataset --force

# Step 2: Unzip the dataset
with zipfile.ZipFile("the-movies-dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("the-movies-dataset_temp")

# Step 3: Move movies_metadata.csv to data folder
os.makedirs("data", exist_ok=True)
shutil.move("the-movies-dataset_temp/movies_metadata.csv", "data/movies_metadata.csv")

# Step 4: Remove unwanted files
for f in ["credits.csv", "keywords.csv", "links.csv", "links_small.csv", "ratings.csv", "ratings_small.csv"]:
    file_path = os.path.join("the-movies-dataset_temp", f)
    if os.path.exists(file_path):
        os.remove(file_path)

# Step 5: Remove zip file
os.remove("the-movies-dataset.zip")

# Step 6: Remove temporary folder
os.rmdir("the-movies-dataset_temp")


Dataset URL: https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset
License(s): CC0-1.0




  0%|          | 0.00/228M [00:00<?, ?B/s]
 18%|█▊        | 41.0M/228M [00:00<00:00, 421MB/s]
 36%|███▌      | 82.0M/228M [00:00<00:00, 340MB/s]
 52%|█████▏    | 119M/228M [00:00<00:00, 357MB/s] 
 78%|███████▊  | 177M/228M [00:00<00:00, 442MB/s]
100%|██████████| 228M/228M [00:00<00:00, 445MB/s]


In [ ]:
def replace_jpg(x):
    return np.nan if isinstance(x, str) and x.endswith('.jpg') else x


tmdb_df = (pd.read_csv("data/movies_metadata.csv")
            .assign(
                    movie_budget = lambda df: df.budget.apply(replace_jpg).astype("Int64").replace(0, pd.NA),
                    movie_revenue_tmdb = lambda df: df.revenue.replace(0.0, pd.NA).astype("Int64")
            )
            .loc[:, ['imdb_id', 'movie_budget', 'movie_revenue_tmdb']]
            .drop_duplicates(subset=['imdb_id'])
          )

C:\Users\khush\AppData\Local\Temp\ipykernel_8340\2615940317.py:5: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  tmdb_df = (pd.read_csv("data/movies_metadata.csv")


In [ ]:
import pandas as pd

# URL for FRED CPI monthly data
cpi_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL"

# Load CSV
cpi_df = pd.read_csv(cpi_url)

# Check column names
print("Columns in CSV:", cpi_df.columns.tolist())


Columns in CSV: ['observation_date', 'CPIAUCSL']


In [ ]:
import pandas as pd

# Load CPI data from FRED
cpi_url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL"
cpi_df = pd.read_csv(cpi_url)

# Rename columns to standard names
cpi_df = cpi_df.rename(columns={"observation_date": "date", "CPIAUCSL": "cpi"})

# Drop missing CPI values
cpi_df = cpi_df.dropna(subset=['cpi'])

# Extract year and compute inflation adjustment
cpi_df = (
    cpi_df
    .assign(
        year=lambda df: df['date'].str[:4].astype(int),
        inflation_adjustment=lambda df: df['cpi'].iloc[-1] / df['cpi'].astype(float)
    )
    .drop(columns=['date', 'cpi'])
    .drop_duplicates(subset=['year'])
    .reset_index(drop=True)
)

print(cpi_df.head())


   year  inflation_adjustment
0  1947             15.100931
1  1948             13.697973
2  1949             13.509704
3  1950             13.797023
4  1951             12.780457


In [ ]:
import requests
import shutil
from pathlib import Path
import gzip

data_path = Path("data")
data_path.mkdir(exist_ok=True)

# Download the IMDb ratings file
url = "https://datasets.imdbws.com/title.ratings.tsv.gz"
gz_path = data_path / "title.ratings.tsv.gz"
tsv_path = data_path / "title.ratings.tsv"

print("Downloading IMDb ratings...")
with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open(gz_path, 'wb') as f:
        shutil.copyfileobj(r.raw, f)

# Unzip the file
print("Unzipping...")
with gzip.open(gz_path, 'rb') as f_in:
    with open(tsv_path, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

# Remove the .gz file
gz_path.unlink()
print("Done. File saved to:", tsv_path)


Unzipping...
Done. File saved to: data\title.ratings.tsv


In [ ]:
import pandas as pd

imdb_df = (pd.read_csv("data/title.ratings.tsv", sep='\t')
             .rename(columns={
                 'tconst': 'imdb_id', 
                 'averageRating': 'imdb_rating', 
                 'numVotes': 'imdb_total_votes'
             })
          )

print(imdb_df.head())


     imdb_id  imdb_rating  imdb_total_votes
0  tt0000001          5.7              2183
1  tt0000002          5.5               304
2  tt0000003          6.4              2263
3  tt0000004          5.2               195
4  tt0000005          6.2              3005


In [ ]:
WIKI_DATA_SERVICE_URL = 'https://query.wikidata.org/sparql'

In [ ]:

# Your SPARQL query
query = '''
SELECT ?movie ?IMDB_ID
WHERE
{
    ?movie wdt:P345 ?IMDB_ID .
}
'''

# Run the query
query_result = requests.get(WIKI_DATA_SERVICE_URL, params={'format': 'json', 'query': query})
wikidata_imdb_df = pd.DataFrame(query_result.json()['results']['bindings'])

# Clean up columns
for column in wikidata_imdb_df.columns:
    wikidata_imdb_df[column] = wikidata_imdb_df[column].apply(lambda x: x['value'] if isinstance(x, dict) and 'value' in x else x)

print(wikidata_imdb_df.head())


                                      movie    IMDB_ID
0  http://www.wikidata.org/entity/Q18636706  tt0071199
1   http://www.wikidata.org/entity/Q2609506  tt0071233
2   http://www.wikidata.org/entity/Q5521259  tt0071527
3  http://www.wikidata.org/entity/Q29581183  tt0072838
4   http://www.wikidata.org/entity/Q7160326  tt0073526


In [ ]:
wikidata_imdb_df = (wikidata_imdb_df
                    .assign(
                        movie_wikidata_id = lambda x: x.movie.str.split('/').str[-1],
                        imdb_id = lambda x: x.IMDB_ID
                    )
                    .loc[:, ['movie_wikidata_id', 'imdb_id']]
                    )

In [ ]:
goodreads_df.columns


Index(['book_publisher', 'book_pages', 'book_rating', 'book_ratings_count',
       'join_title', 'join_author'],
      dtype='object')

In [ ]:
tmdb_df.columns

Index(['imdb_id', 'movie_revenue_tmdb'], dtype='object')

In [ ]:
cmu_df.columns

Index(['movie_title', 'imdb_id', 'movie_budget', 'movie_revenue',
       'movie_release'],
      dtype='object')

In [ ]:
# Convert movie_budget and movie_revenue to numeric, invalid parsing becomes NaN
cmu_df['movie_budget'] = pd.to_numeric(cmu_df['movie_budget'], errors='coerce')
cmu_df['movie_revenue'] = pd.to_numeric(cmu_df['movie_revenue'], errors='coerce')

# If TMDB revenue exists, also convert it
if 'movie_revenue_tmdb' in tmdb_df.columns:
    tmdb_df['movie_revenue_tmdb'] = pd.to_numeric(tmdb_df['movie_revenue_tmdb'], errors='coerce')


In [ ]:
book_adaptation_df = (
    cmu_df
    .merge(goodreads_df, on='join_title', how='left')
    .merge(imdb_df, on='imdb_id', how='left')
    .merge(tmdb_df, on='imdb_id', how='left')
    .merge(cpi_df, left_on='movie_release', right_on='year', how='left')
    .assign(
        movie_budget=lambda df: df.movie_budget * df.inflation_adjustment,
        movie_revenue=lambda df: (df.movie_revenue.fillna(df.movie_revenue_tmdb) * df.inflation_adjustment)
    )
    .drop(columns=['year', 'inflation_adjustment', 'movie_revenue_tmdb'])
)


In [ ]:
# ------------------------------
# 1. Mark movies as adaptations
# ------------------------------
# If you have a column from Goodreads or Wikidata that indicates a book link, use it.
# If not, you could mark movies that successfully matched a Goodreads entry as an adaptation
book_adaptation_df['movie_is_adaptation'] = book_adaptation_df['join_author'].notna()

# ------------------------------
# 2. Drop any unnecessary columns
# ------------------------------
# Columns you used for merging or intermediate calculations
columns_to_drop = ['join_title', 'join_author', 'movie_revenue_tmdb', 'year', 'inflation_adjustment']
book_adaptation_df = book_adaptation_df.drop(columns=[col for col in columns_to_drop if col in book_adaptation_df.columns])

# ------------------------------
# 3. Quick check
# ------------------------------
print(book_adaptation_df.info())
print(book_adaptation_df.head())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   movie_title          45460 non-null  object 
 1   imdb_id              45449 non-null  object 
 2   movie_budget         42276 non-null  float64
 3   movie_revenue        42276 non-null  float64
 4   movie_release        45376 non-null  float64
 5   book_publisher       0 non-null      object 
 6   book_pages           0 non-null      Int64  
 7   book_rating          0 non-null      float64
 8   book_ratings_count   0 non-null      Int64  
 9   imdb_rating          45380 non-null  float64
 10  imdb_total_votes     45380 non-null  float64
 11  movie_is_adaptation  45466 non-null  bool   
dtypes: Int64(2), bool(1), float64(6), object(3)
memory usage: 3.9+ MB
None
                   movie_title    imdb_id  movie_budget  movie_revenue  \
0                    Toy Story  tt0114709

In [ ]:
# Fill missing revenue from TMDB if column exists
if 'movie_revenue_tmdb' in book_adaptation_df.columns:
    book_adaptation_df['movie_revenue'] = book_adaptation_df['movie_revenue'].fillna(book_adaptation_df['movie_revenue_tmdb'])
else:
    print("No TMDB revenue column found, skipping.")

# Fill missing budget from TMDB if column exists
if 'movie_budget_tmdb' in book_adaptation_df.columns:
    book_adaptation_df['movie_budget'] = book_adaptation_df['movie_budget'].fillna(book_adaptation_df['movie_budget_tmdb'])
else:
    print("No TMDB budget column found, skipping.")


No TMDB revenue column found, skipping.
No TMDB budget column found, skipping.


In [ ]:
if 'movie_revenue_tmdb' in book_adaptation_df.columns:
    book_adaptation_df['movie_revenue'] = book_adaptation_df['movie_revenue'].fillna(book_adaptation_df['movie_revenue_tmdb'])


In [ ]:
# ------------------------------
# 1. Prepare join keys
# ------------------------------
cmu_df = cmu_df.assign(
    join_title=lambda df: df.movie_title.str.lower().str.strip()
)

goodreads_df = goodreads_df.assign(
    join_title=lambda df: df.join_title.str.lower().str.strip(),
    join_author=lambda df: df.join_author.str.lower().str.strip()
)

# ------------------------------
# 2. Merge datasets
# ------------------------------
book_adaptation_df = (
    cmu_df
    # Merge Goodreads on title
    .merge(goodreads_df, on='join_title', how='left')
    # Merge IMDb ratings
    .merge(imdb_df, on='imdb_id', how='left')
    # Merge TMDB revenue (only revenue available)
    .merge(tmdb_df, on='imdb_id', how='left')
    # Merge CPI for inflation adjustment
    .merge(cpi_df, left_on='movie_release', right_on='year', how='left')
)

# ------------------------------
# 3. Adjust budget and revenue for inflation
# ------------------------------
# Convert columns to numeric if they exist
for col in ['movie_budget', 'movie_revenue', 'movie_revenue_tmdb']:
    if col in book_adaptation_df.columns:
        book_adaptation_df[col] = pd.to_numeric(book_adaptation_df[col], errors='coerce')

# Fill missing revenue from TMDB if movie_revenue is NaN
if 'movie_revenue_tmdb' in book_adaptation_df.columns:
    book_adaptation_df['movie_revenue'] = book_adaptation_df['movie_revenue'].fillna(
        book_adaptation_df['movie_revenue_tmdb']
    )

# Adjust for inflation if CPI column exists
if 'inflation_adjustment' in book_adaptation_df.columns:
    if 'movie_budget' in book_adaptation_df.columns:
        book_adaptation_df['movie_budget'] = book_adaptation_df['movie_budget'] * book_adaptation_df['inflation_adjustment']
    if 'movie_revenue' in book_adaptation_df.columns:
        book_adaptation_df['movie_revenue'] = book_adaptation_df['movie_revenue'] * book_adaptation_df['inflation_adjustment']

# ------------------------------
# 4. Drop unnecessary columns
# ------------------------------
drop_cols = ['year', 'inflation_adjustment', 'movie_revenue_tmdb']
book_adaptation_df = book_adaptation_df.drop(columns=[c for c in drop_cols if c in book_adaptation_df.columns])


In [ ]:
import pandas as pd
import numpy as np

# Replace NaN in numeric columns with 0
numeric_cols = book_adaptation_df.select_dtypes(include=[np.number]).columns
book_adaptation_df[numeric_cols] = book_adaptation_df[numeric_cols].fillna(0)

# Save as CSV
book_adaptation_df.to_csv("book_adaptation_df_clean.csv", index=False)

# Optionally, save as Excel
book_adaptation_df.to_excel("book_adaptation_df_clean.xlsx", index=False)

print("Dataset saved successfully!")


Dataset saved successfully!
